In [9]:
from tools.influence_analysis_helpers import visualize_sentence_similarity_timeline
# Create interactive output
from ipywidgets import interactive, IntSlider, FloatSlider
import pandas as pd
import numpy as np

In [10]:
clause_type = "modification"  # privacy, liability, termination, indemnity, warranty
influence_scores = pd.read_csv(f"results/tous/influence_scores_{clause_type}.tsv" , sep="\t")
nodes_df = pd.read_csv(f"results/tous/nodes_{clause_type}.tsv" , sep="\t")
emb_norm = np.loadtxt(f'results/tous/emb_norm_{clause_type}.txt', delimiter=',') 

In [11]:


# Get top influential sentence IDs
top_ids = influence_scores.nlargest(1000, 'influence_score')['node_id'].values

sentence_idx_s = IntSlider(value=int(top_ids[0]), min=0, max=len(nodes_df)-1, step=1, description='Sentence:', layout={'width': '700px'})
min_sim_val = FloatSlider(value=0.80, min=0.50, max=0.99, step=0.01, description='Min Sim:', layout={'width': '400px'})

def show_timeline(sent_idx, min_sim):
    row = nodes_df.iloc[sent_idx]
    infl_score = influence_scores.iloc[sent_idx]['influence_score']
    
    print(f"\n{'='*100}")
    print(f"ROOT SENTENCE: {row['platform'].upper()} ({int(row['year'])}) | Influence Score: {infl_score:.2f}")
    print(f"{'='*100}")
    print(f"{row['sentence'][:400]}{'...' if len(row['sentence']) > 400 else ''}\n")
    
    fig, res = visualize_sentence_similarity_timeline(sent_idx, nodes_df, emb_norm, min_similarity=min_sim)
    if fig:
        print(f"Found {len(res)-1} similar sentences with similarity >= {min_sim:.2f}\n")
        fig.show()
    else:
        print(f"No similar sentences found with similarity >= {min_sim:.2f}\n")

interactive_plot = interactive(show_timeline, sent_idx=sentence_idx_s, min_sim=min_sim_val)
display(interactive_plot)

interactive(children=(IntSlider(value=3356, description='Sentence:', layout=Layout(width='700px'), max=6248), …